# Stage 4a — ETL Orchestration
**MSc Marketing Thesis | VU Amsterdam | Alexandru Constantinescu**

---

This notebook is the glue layer between data collection (Stages 1-2) and analysis (Stages 5-7). It moves raw Bronze data from Google Cloud Storage (GCS) into BigQuery, applies Python-level cleaning that cannot be expressed in SQL, and triggers the SQL transforms that produce analysis-ready Silver and Gold tables.

Run cells top-to-bottom for a full pipeline pass. Individual sections can be re-run independently when iterating on SQL.

---

## Pipeline overview

```
GCS Bronze                     BigQuery
──────────────────             ─────────────────────────────────────
bronze/reddit/{event}/*.jsonl  ->  bronze_reddit
bronze/amazon/{event}.csv      ->  bronze_amazon
                                       │
                               Silver SQL transforms
                               (dedup · date windows · word count)
                                       │
                               silver_reddit  silver_amazon
                                       │
                               Gold SQL aggregation
                                       │
                               gold_pre_launch  gold_post_launch
                                       │
                               -> Stage 5 (LLM extraction)
```


---
## Cell 1 — Imports and configuration

All project-level constants are defined here. Changing a value in this cell propagates to the entire notebook without touching downstream logic.

**Authentication:** The BigQuery and GCS clients authenticate via the `GOOGLE_APPLICATION_CREDENTIALS` environment variable, which must point to the service account JSON key stored outside the project root. This follows GCP best practice — the key is never committed to version control.

**Why `WRITE_APPEND` in BigQuery?** Bronze tables accumulate data across multiple events. Deduplication is handled downstream by the Silver SQL transforms (via `ROW_NUMBER()` window functions partitioned by `post_id` / `review_id`). This keeps the load step simple and idempotent — re-running for a single event appends, and Silver SQL cleans up.

In [88]:
import io
import json
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from google.cloud import bigquery, storage
from google.cloud.bigquery import LoadJobConfig, SchemaField, WriteDisposition
from langdetect import detect, LangDetectException

import hashlib
import io
import json

import google.auth
import pandas as pd
from google.cloud import bigquery as bq_module, storage as gcs_module

import hashlib

# ── GCP constants ─────────────────────────────────────────────────────────────
PROJECT_ID  = "vuthesis-llm-buzz"
BUCKET_NAME = "thesis-bucket-vua"
DATASET_ID  = "thesis_dataset"

# ── Local paths ───────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("C:/Users/User/Desktop/VU/Thesis/Code/LLM-prelaunch-buzz-postlaunch-predictor")
EVENTS_PATH  = PROJECT_ROOT / "final_events.csv"
SQL_DIR      = PROJECT_ROOT / "sql"
LOG_DIR      = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── BigQuery fully-qualified table names ──────────────────────────────────────
BQ_BRONZE_REDDIT = f"{PROJECT_ID}.{DATASET_ID}.bronze_reddit"
BQ_BRONZE_AMAZON = f"{PROJECT_ID}.{DATASET_ID}.bronze_amazon"
BQ_SILVER_REDDIT = f"{PROJECT_ID}.{DATASET_ID}.silver_reddit"
BQ_SILVER_AMAZON = f"{PROJECT_ID}.{DATASET_ID}.silver_amazon"
BQ_GOLD_PRE      = f"{PROJECT_ID}.{DATASET_ID}.gold_pre_launch"
BQ_GOLD_POST     = f"{PROJECT_ID}.{DATASET_ID}.gold_post_launch"

# ── Minimum thresholds (from research design) ────────────────────────────────
MIN_PRE_LAUNCH  = 500   # Reddit posts per event
MIN_POST_LAUNCH = 200   # Amazon reviews per event

# ── Required columns for validation ──────────────────────────────────────────
REQUIRED_COLS_REDDIT = {"post_id", "product_event", "body_text"}
REQUIRED_COLS_AMAZON = {"review_id", "product_event", "review_text"}

print("✓ Configuration loaded.")

✓ Configuration loaded.


---
## Cell 2 — BigQuery schemas

BigQuery schemas are defined explicitly rather than using autodetect. Autodetect infers types from the first few hundred rows and fails silently on sparse or mixed-type columns — a known failure mode for user-generated text data where fields like `verified_purchase` or `upvotes` are frequently null.

**Key schema decisions:**

| Field | Decision | Rationale |
|---|---|---|
| `comments` (Reddit) | `STRING` | Top-level comments are stored as a JSON array serialised to string. BigQuery's `JSON` type would require a minimum API version — STRING is safer and sufficient for LLM input. |
| `created_utc` | `TIMESTAMP` | UTC Unix timestamps from the Reddit scraper are cast to TIMESTAMP in the Silver SQL for clean date arithmetic. |
| `verified_purchase` | `BOOLEAN` | Retained from Amazon metadata. Used as a control variable in robustness checks (Stage 7). |
| `days_to_launch` / `days_since_launch` | `INTEGER` | Pre-computed in the collection scripts. Silver SQL validates and re-computes from `launch_date` as a cross-check. |

Both `post_id` and `review_id` are marked `REQUIRED`. A row without an identifier cannot be deduplicated downstream and must not enter the pipeline.

In [89]:
print("═" * 65)
print("CELL 2 — Bronze Bridge: BigQuery → GCS Bronze (JSONL)")
print("═" * 65)
print("Scope:     14 threshold-passing events (final_events.csv)")
print("Source:    thesis_data.phones_reviews + electronics_reviews")
print("Writes to: gs://thesis-bucket-vua/bronze/amazon/{event}.jsonl")
print()

# ── Config ────────────────────────────────────────────────────────────────────
BRIDGE_PROJECT = "vuthesis-llm-buzz"
BRIDGE_BUCKET  = "thesis-bucket-vua"
BQ_PHONES      = f"{BRIDGE_PROJECT}.thesis_data.phones_reviews"
BQ_ELECTRONICS = f"{BRIDGE_PROJECT}.thesis_data.electronics_reviews"
WINDOW_MIN, WINDOW_MAX = 1, 90

# ── Slug → BQ model name (14 final events only) ───────────────────────────────
SLUG_TO_MODEL = {
    # ── Hedonic ───────────────────────────────────────────────────────────────
    "ipad_air_2":       "iPad Air 2",          # electronics_reviews
    "ipad_pro_2018":    "iPad Pro 2018",        # electronics_reviews
    "ipad_air_m1":      "iPad Air M1",          # electronics_reviews
    "galaxy_s6":        "Samsung Galaxy S6",    # phones_reviews
    "galaxy_s20_fe":    "Samsung Galaxy S20 FE",# phones_reviews
    "galaxy_s21_ultra": "Samsung Galaxy S21 Ultra", # phones_reviews
    "galaxy_s22_ultra": "Samsung Galaxy S22 Ultra", # phones_reviews
    "lg_g3":            "LG G3",               # phones_reviews

    # ── Utilitarian ───────────────────────────────────────────────────────────
    "moto_g_3rd_gen":   "Moto G 3rd Gen",      # phones_reviews
    "moto_g_4th_gen":   "Moto G 4th Gen",      # phones_reviews
    "moto_g_fast":      "Moto G Fast",          # phones_reviews
    "pixel_3a":         "Google Pixel 3a",      # phones_reviews
    "pixel_4a":         "Google Pixel 4a",      # phones_reviews
    "galaxy_tab_s2":    "Galaxy Tab S2",        # electronics_reviews
}

# ── Column rename: BQ schema → Bronze schema ──────────────────────────────────
RENAME = {
    "text":      "review_text",
    "timestamp": "review_date",
    "model":     "product_name",
}

# ── Auth ──────────────────────────────────────────────────────────────────────
creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
bq_client  = bq_module.Client(project=BRIDGE_PROJECT, credentials=creds)
gcs_client = gcs_module.Client(project=BRIDGE_PROJECT, credentials=creds)


# ── Helpers ───────────────────────────────────────────────────────────────────

def make_review_id(row) -> str:
    """Stable 16-char SHA1 from asin + timestamp + text[:50]."""
    key = f"{row.get('asin','')}{row.get('review_date','')}{str(row.get('review_text',''))[:50]}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


def fetch_reviews(model_name: str) -> pd.DataFrame:
    """
    Pull window-filtered reviews for one model from both BQ tables.
    Uses parameterised query to prevent SQL injection on model name.
    """
    query = """
        SELECT asin, rating, text, timestamp, helpful_vote,
               verified_purchase, model, days_since_launch
        FROM `{phones}`
        WHERE model = @model
          AND days_since_launch BETWEEN @wmin AND @wmax

        UNION ALL

        SELECT asin, rating, text, timestamp, helpful_vote,
               verified_purchase, model, days_since_launch
        FROM `{electronics}`
        WHERE model = @model
          AND days_since_launch BETWEEN @wmin AND @wmax
    """.format(phones=BQ_PHONES, electronics=BQ_ELECTRONICS)

    job_config = bq_module.QueryJobConfig(query_parameters=[
        bq_module.ScalarQueryParameter("model", "STRING", model_name),
        bq_module.ScalarQueryParameter("wmin",  "INT64",  WINDOW_MIN),
        bq_module.ScalarQueryParameter("wmax",  "INT64",  WINDOW_MAX),
    ])
    return bq_client.query(query, job_config=job_config).to_dataframe()


def transform(df: pd.DataFrame, slug: str) -> pd.DataFrame:
    """Rename columns to Bronze schema, add review_id and product_event."""
    df = df.rename(columns=RENAME).copy()

    # Normalise date to plain string
    df["review_date"] = pd.to_datetime(df["review_date"]).dt.date.astype(str)

    # Add pipeline fields
    df["product_event"] = slug
    df["review_id"]     = df.apply(make_review_id, axis=1)

    # Final column order
    cols = [
        "review_id", "asin", "product_name", "product_event",
        "rating", "review_text", "verified_purchase",
        "review_date", "days_since_launch",
    ]
    return df[[c for c in cols if c in df.columns]]


def upload_jsonl(df: pd.DataFrame, slug: str) -> str:
    """
    Serialise DataFrame as newline-delimited JSON and upload to GCS Bronze.
    Consistent with Reddit Bronze format — orchestrate.ipynb Cell 10 reads JSONL.
    Returns gs:// URI.
    """
    lines     = "\n".join(row.to_json() for _, row in df.iterrows())
    blob_path = f"bronze/amazon/{slug}.jsonl"
    blob      = gcs_client.bucket(BRIDGE_BUCKET).blob(blob_path)
    blob.upload_from_string(lines, content_type="application/json")
    return f"gs://{BRIDGE_BUCKET}/{blob_path}"


# ── Main loop ─────────────────────────────────────────────────────────────────
results = []

for slug, model_name in SLUG_TO_MODEL.items():
    print(f"  {slug:<28} ← '{model_name}'")

    try:
        df_raw = fetch_reviews(model_name)
    except Exception as e:
        print(f"    ✗ BQ query failed: {e}")
        results.append((slug, model_name, 0, "BQ_ERROR"))
        continue

    if df_raw.empty:
        print(f"    ⚠ Not found in BQ — skipping")
        results.append((slug, model_name, 0, "NOT_IN_BQ"))
        continue

    df_bronze = transform(df_raw, slug)

    try:
        uri = upload_jsonl(df_bronze, slug)
        print(f"    ✓ {len(df_bronze):>5} rows → {uri}")
        results.append((slug, model_name, len(df_bronze), "OK"))
    except Exception as e:
        print(f"    ✗ Upload failed: {e}")
        results.append((slug, model_name, 0, "UPLOAD_ERROR"))

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("═" * 65)
results_df = pd.DataFrame(results, columns=["slug", "bq_model", "rows", "status"])

ok        = (results_df["status"] == "OK").sum()
not_in_bq = (results_df["status"] == "NOT_IN_BQ").sum()
errors    = results_df["status"].isin(["BQ_ERROR", "UPLOAD_ERROR"]).sum()

print(f"  ✓ Written:    {ok} JSONL files")
print(f"  ⚠ Not in BQ: {not_in_bq} events")
print(f"  ✗ Errors:    {errors} events")
print()

if not_in_bq > 0:
    missing = results_df[results_df["status"] == "NOT_IN_BQ"]
    print("  Events not found in BQ — check model name spelling:")
    for _, r in missing.iterrows():
        print(f"    • {r['slug']:<28} model='{r['bq_model']}'")
    print()

print("  ⚠ NOTE: Cell 10 (load_amazon_bronze) reads .jsonl — not .csv.")
print("  If it still references .csv, update the blob_path and use json.loads().")
print()

def colour(v):
    return {
        "OK":           "background-color:#d4edda;color:#155724",
        "NOT_IN_BQ":    "background-color:#fff3cd;color:#856404",
        "BQ_ERROR":     "background-color:#f8d7da;color:#721c24",
        "UPLOAD_ERROR": "background-color:#f8d7da;color:#721c24",
    }.get(v, "")

display(
    results_df.style
    .applymap(colour, subset=["status"])
    .format({"rows": "{:,}"})
    .set_caption("Cell 2 — Bronze bridge results (JSONL)")
)

═════════════════════════════════════════════════════════════════
CELL 2 — Bronze Bridge: BigQuery → GCS Bronze (JSONL)
═════════════════════════════════════════════════════════════════
Scope:     14 threshold-passing events (final_events.csv)
Source:    thesis_data.phones_reviews + electronics_reviews
Writes to: gs://thesis-bucket-vua/bronze/amazon/{event}.jsonl

  ipad_air_2                   ← 'iPad Air 2'


c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    ✓   254 rows → gs://thesis-bucket-vua/bronze/amazon/ipad_air_2.jsonl
  ipad_pro_2018                ← 'iPad Pro 2018'


c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    ✓   371 rows → gs://thesis-bucket-vua/bronze/amazon/ipad_pro_2018.jsonl
  ipad_air_m1                  ← 'iPad Air M1'


...

    ✓   334 rows → gs://thesis-bucket-vua/bronze/amazon/pixel_4a.jsonl
  galaxy_tab_s2                ← 'Galaxy Tab S2'


c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    ✓  1313 rows → gs://thesis-bucket-vua/bronze/amazon/galaxy_tab_s2.jsonl

═════════════════════════════════════════════════════════════════
  ✓ Written:    14 JSONL files
  ⚠ Not in BQ: 0 events
  ✗ Errors:    0 events

  ⚠ NOTE: Cell 10 (load_amazon_bronze) reads .jsonl — not .csv.
  If it still references .csv, update the blob_path and use json.loads().



C:\Users\User\AppData\Local\Temp\ipykernel_14312\3808767864.py:185: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(colour, subset=["status"])


,slug,bq_model,rows,status
0,ipad_air_2,iPad Air 2,254,OK
1,ipad_pro_2018,iPad Pro 2018,371,OK
2,ipad_air_m1,iPad Air M1,246,OK
3,galaxy_s6,Samsung Galaxy S6,233,OK
4,galaxy_s20_fe,Samsung Galaxy S20 FE,240,OK
5,galaxy_s21_ultra,Samsung Galaxy S21 Ultra,217,OK
6,galaxy_s22_ultra,Samsung Galaxy S22 Ultra,280,OK
7,lg_g3,LG G3,223,OK
8,moto_g_3rd_gen,Moto G 3rd Gen,365,OK
9,moto_g_4th_gen,Moto G 4th Gen,402,OK


In [34]:
SCHEMA_BRONZE_REDDIT = [
    SchemaField("post_id",        "STRING",    mode="REQUIRED"),
    SchemaField("product_event",  "STRING",    mode="REQUIRED"),
    SchemaField("subreddit",      "STRING",    mode="NULLABLE"),
    SchemaField("title",          "STRING",    mode="NULLABLE"),
    SchemaField("body_text",      "STRING",    mode="NULLABLE"),
    SchemaField("comments",       "STRING",    mode="NULLABLE"),  # JSON array as string
    SchemaField("upvotes",        "INTEGER",   mode="NULLABLE"),
    SchemaField("created_utc",    "TIMESTAMP", mode="NULLABLE"),
    SchemaField("days_to_launch", "FLOAT",   mode="NULLABLE"),
]

SCHEMA_BRONZE_AMAZON = [
    SchemaField("review_id",         "STRING",  mode="REQUIRED"),
    SchemaField("asin",              "STRING",  mode="NULLABLE"),
    SchemaField("product_name",      "STRING",  mode="NULLABLE"),
    SchemaField("product_event",     "STRING",  mode="REQUIRED"),
    SchemaField("rating",            "FLOAT",   mode="NULLABLE"),
    SchemaField("review_text",       "STRING",  mode="NULLABLE"),
    SchemaField("verified_purchase", "BOOLEAN", mode="NULLABLE"),
    SchemaField("review_date",       "DATE",    mode="NULLABLE"),
    SchemaField("days_since_launch", "INTEGER", mode="NULLABLE"),
]

print("✓ Schemas defined.")

✓ Schemas defined.


---
## Cell 3 — Logging and GCP clients

The logging setup writes simultaneously to a timestamped file in `/logs/` and to stdout (this notebook's output). After the run, the log file is uploaded to `gs://thesis-bucket/logs/` for a permanent audit trail — important for thesis reproducibility documentation.

The log captures: row counts at every transformation step, rows removed by each filter, BigQuery job outcomes, and threshold check results per event.

In [91]:
_RUN_TS       = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE_PATH = LOG_DIR / f"orchestrate_{_RUN_TS}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE_PATH, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger(__name__)

# ── GCP clients ───────────────────────────────────────────────────────────────
# Credentials are read from GOOGLE_APPLICATION_CREDENTIALS env var.
# Set it before running:
#   set GOOGLE_APPLICATION_CREDENTIALS=C:\Users\User\.gcp\thesis-sa-key.json
creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

gcs_client = storage.Client(project=PROJECT_ID, credentials=creds)
bq_client  = bigquery.Client(project=PROJECT_ID, credentials=creds)

log.info("Clients initialised (project=%s)", PROJECT_ID)

2026-05-25 13:47:49  INFO      Clients initialised (project=vuthesis-llm-buzz)


---
## Cell 4 — Load product events

`events.csv` is the single source of truth for which product launches are in scope. It was defined manually in Stage 0 based on three inclusion criteria:

1. **Confirmed public launch date** — needed to define the 90-day pre-launch and 60-day post-launch collection windows
2. **≥ 500 Reddit posts** in the pre-launch window — minimum discourse volume for reliable signal extraction
3. **≥ 200 Amazon reviews** in the post-launch window — minimum review volume for stable sentiment estimation

The `product_type` column encodes the hedonic/utilitarian moderator (1 = hedonic, 0 = utilitarian). This value is fixed here and carried through to the modelling dataframe in Stage 6.

**To filter to a single event for testing**, set `TARGET_EVENT` to the event name. Set to `None` to process all events.

In [92]:
# Set to a specific product_event name for single-event testing, or None for all
TARGET_EVENT = None  # e.g. "iphone16"

def load_events(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"events.csv not found at {path}")

    df = pd.read_csv(path, parse_dates=["launch_date"])

    required = {"product_event", "launch_date", "product_type"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"events.csv missing required columns: {missing}")

    log.info("Loaded %d product events from events.csv", len(df))
    return df

events_df = load_events(EVENTS_PATH)

if TARGET_EVENT:
    events_df = events_df[events_df["product_event"] == TARGET_EVENT]
    if events_df.empty:
        raise ValueError(f"TARGET_EVENT '{TARGET_EVENT}' not found in events.csv")
    log.info("Single-event mode: %s", TARGET_EVENT)

events_df[["product_event", "launch_date", "product_type"]]

2026-05-25 13:47:56  INFO      Loaded 14 product events from events.csv


,product_event,launch_date,product_type
0,ipad_air_2,2014-10-16,1
1,ipad_pro_2018,2018-11-07,1
2,ipad_air_m1,2022-03-18,1
3,galaxy_s6,2015-04-10,1
4,galaxy_s20_fe,2020-10-02,1
5,galaxy_s21_ultra,2021-01-29,1
6,galaxy_s22_ultra,2022-02-25,1
7,lg_g3,2014-07-18,1
8,moto_g_3rd_gen,2015-07-28,0
9,moto_g_4th_gen,2016-05-17,0


---
## Cell 5 — Language filtering

**Why Python, not SQL?** `langdetect` — the standard library for language identification — is a Python package. It cannot be called from inside BigQuery SQL. Language filtering must therefore happen before the data enters BigQuery, making this one of the few cleaning steps that lives in Python rather than SQL.

**How `langdetect` works:** The library uses a Naive Bayes classifier trained on character n-gram profiles for 55 languages. It returns a language code (`'en'`, `'nl'`, `'de'`, etc.) with a confidence score. For short texts the classifier is unreliable, so texts shorter than 20 characters are passed through unconditionally — the Silver SQL word-count filter (< 10 words) handles them downstream.

**Design choice — keep on detection failure:** When `langdetect` raises a `LangDetectException` (e.g. text is purely numeric, emoji-only, or too noisy), the record is kept rather than dropped. Silent discard on error introduces a non-random exclusion pattern that could bias the corpus.

**Expected removal rate:** Consumer electronics Reddit threads are predominantly English-language. Typical removal rates are < 3% for r/apple and r/GooglePixel. Rates above 10% for a given event should be flagged for manual inspection.

In [93]:
def _is_english(text: str) -> bool:
    """
    Return True if text is detected as English.
    Texts < 20 characters bypass classification — too short for reliable detection.
    On LangDetectException, the record is kept (fail-open).
    """
    if not isinstance(text, str) or len(text.strip()) < 20:
        return True
    try:
        return detect(text) == "en"
    except LangDetectException:
        return True


def filter_language(
    df: pd.DataFrame,
    text_col: str,
    event: str,
    source: str,
) -> pd.DataFrame:
    """Remove non-English rows. Logs before/after counts."""
    n_before  = len(df)
    df_clean  = df[df[text_col].apply(_is_english)].copy()
    n_removed = n_before - len(df_clean)
    log.info(
        "[%s][%s] Language filter: removed %d of %d rows (%.1f%%)",
        event, source, n_removed, n_before,
        100 * n_removed / n_before if n_before > 0 else 0,
    )
    return df_clean

print("✓ Language filter functions defined.")

✓ Language filter functions defined.


---
## Cell 6 — Schema validation

Schema validation is a guard between the collection scripts and the BigQuery load. Collection scripts can silently change output structure (e.g. a Reddit API field being renamed or removed). Without validation, the BigQuery load job fails with an opaque type error mid-pipeline.

**Three-tier severity:**

| Condition | Action | Rationale |
|---|---|---|
| Required column missing entirely | Skip event, log `ERROR` | Data is structurally unusable |
| Required column 100% null | Skip event, log `ERROR` | Column exists in header but carries no data |
| Required column > 30% null | Continue, log `WARNING` | Partial data is still usable; flag for manual check |

The 30% threshold for warnings is deliberately conservative. For `body_text` in Reddit data, high null rates typically mean the scraper captured post metadata but not content (e.g. link posts with no body). These records will be removed by the Silver SQL short-text filter anyway.

In [94]:
def validate_schema(
    df: pd.DataFrame,
    required_cols: set,
    event: str,
    source: str,
) -> bool:
    """
    Validate that required columns exist and contain data.
    Returns True (valid) or False (skip this event/source).
    """
    missing = required_cols - set(df.columns)
    if missing:
        log.error(
            "[%s][%s] Schema FAILED — missing columns: %s. Skipping.",
            event, source, missing,
        )
        return False

    for col in required_cols:
        null_rate = df[col].isnull().mean()
        if null_rate == 1.0:
            log.error(
                "[%s][%s] Column '%s' is entirely null. Skipping.",
                event, source, col,
            )
            return False
        if null_rate > 0.30:
            log.warning(
                "[%s][%s] Column '%s' is %.0f%% null — check collection script.",
                event, source, col, null_rate * 100,
            )

    log.info("[%s][%s] Schema validation passed (%d rows).", event, source, len(df))
    return True

print("✓ Schema validation function defined.")

✓ Schema validation function defined.


---
## Cell 7 - GCS helpers

Two utility functions for interacting with GCS:

**`list_blobs()`** - lists all objects under a GCS prefix. Reddit Bronze data is sharded by date (`bronze/reddit/{event}/{date}.jsonl`) because the scraper streamed daily batches. The orchestrator consolidates all shards into a single filtered file per event before the BigQuery load.

**`upload_df_to_gcs()`** - serialises a DataFrame to JSONL or CSV and uploads directly to GCS using `upload_from_string()`. No local temp file is written to disk. This keeps the Anaconda environment clean and avoids Windows path-length issues with large temp files.

**Why re-upload filtered data back to GCS Bronze?** BigQuery's `bq load` reads from GCS, not from Python memory. The re-upload step creates the `filtered.jsonl` / `_filtered.csv` that `bq load` then reads. Original raw files (`{date}.jsonl`, `{event}.csv`) are never overwritten — Bronze data is immutable by design.

In [95]:
def list_blobs(prefix: str) -> list:
    """Return all blob objects under a GCS prefix."""
    return list(gcs_client.bucket(BUCKET_NAME).list_blobs(prefix=prefix))


def upload_df_to_gcs(df: pd.DataFrame, blob_path: str, fmt: str) -> str:
    """
    Serialise df to JSONL or CSV and upload to GCS without writing a local temp file.
    Returns the full gs:// URI.

    fmt: 'jsonl' for Reddit (line-delimited JSON) | 'csv' for Amazon
    """
    blob = gcs_client.bucket(BUCKET_NAME).blob(blob_path)

    if fmt == "jsonl":
        lines = "\n".join(df.apply(lambda r: r.to_json(), axis=1))
        blob.upload_from_string(lines, content_type="application/json")
    elif fmt == "csv":
        blob.upload_from_string(df.to_csv(index=False), content_type="text/csv")
    else:
        raise ValueError(f"Unknown fmt: {fmt!r}. Expected 'jsonl' or 'csv'.")

    uri = f"gs://{BUCKET_NAME}/{blob_path}"
    log.info("Uploaded %d rows → %s", len(df), uri)
    return uri

print("✓ GCS helper functions defined.")

✓ GCS helper functions defined.


---
## Cell 8 - BigQuery load function

The `bq_load()` function loads a GCS file into a BigQuery table using the official BigQuery Python client.

**Key configuration choices:**

| Parameter | Value | Rationale |
|---|---|---|
| `write_disposition` | `WRITE_APPEND` | Bronze tables accumulate across events. Silver SQL deduplicates. |
| `autodetect` | `False` | Explicit schema prevents silent type mismatches on sparse columns. |
| `max_bad_records` | `0` | Zero tolerance for malformed rows. Fail loudly and log the error so the source can be fixed. |
| `ignore_unknown_values` | `False` | Extra columns in the source file indicate the collection script has changed — surface it immediately. |
| `skip_leading_rows` | `1` for CSV, `0` for JSONL | CSVs have a header row; JSONL files do not. |

**`job.result()`** blocks execution until BigQuery confirms the load job is complete. This is intentional — the Silver SQL transforms in the next cell depend on the Bronze tables being fully populated. Async execution here would create a race condition.

In [96]:
def bq_load(
    source_uri: str,
    destination_table: str,
    schema: list,
    source_format: bigquery.SourceFormat,
    event: str,
    source: str,
) -> int:
    """
    Load a GCS file into a BigQuery table (WRITE_APPEND, explicit schema).
    Blocks until the load job completes.
    Returns the number of rows appended, or 0 on failure.
    """
    job_config = LoadJobConfig(
        schema=schema,
        source_format=source_format,
        write_disposition=WriteDisposition.WRITE_APPEND,
        autodetect=False,
        ignore_unknown_values=True,
        max_bad_records=0,
    )

    log.info("[%s][%s] bq load: %s → %s", event, source, source_uri, destination_table)
    try:
        job = bq_client.load_table_from_uri(
            source_uri, destination_table, job_config=job_config
        )
        job.result()  # Block until complete — Silver SQL depends on this finishing first
    except Exception as exc:
        log.error("[%s][%s] bq load FAILED: %s", event, source, exc)
        return 0

    n_appended = job.output_rows
    table_rows = bq_client.get_table(destination_table).num_rows
    log.info(
        "[%s][%s] Load complete: appended=%d | table_total=%d",
        event, source, n_appended, table_rows,
    )
    return n_appended

print("✓ BigQuery load function defined.")

✓ BigQuery load function defined.


---
## Cell 9 - Reddit Bronze processing

This cell handles the full Bronze processing pipeline for Reddit data.

**Blob consolidation:** The Reddit scraper wrote daily JSONL shards (`bronze/reddit/{event}/2024-08-01.jsonl`, `2024-08-02.jsonl`, ...) because it streamed batches to GCS in real time. The orchestrator downloads all shards, parses them, concatenates, and re-uploads a single `filtered.jsonl` per event. This produces one clean GCS object for the `bq load` call, rather than requiring BigQuery to handle a wildcard URI (`bronze/reddit/{event}/*.jsonl`) - the simpler approach.

**Line-by-line JSON parsing:** Each line in the JSONL files is parsed individually with a `try/except`. A malformed line (e.g. a truncated write from a connection drop) logs a warning and is skipped. The remaining lines in the file are still processed - a single bad line does not discard the entire shard.

**Filtered blob naming:** Daily shard blobs are excluded from the load by filtering out any blob whose name contains `'filtered'`. This prevents double-loading if the cell is re-run.

In [ ]:
def load_reddit_bronze(event: str) -> pd.DataFrame:
    # Multi-folder mapping - same as Check 8
    gcs_folder_mapping = {
        "macbook_air_m1":   ["macbook_air_m1_(2020)"],
        "macbook_pro_16":   ['macbook_pro_16"'],
        "ipad_7th_gen":     ["ipad_7th_generation"],
        "ipad_9th_gen":     ["ipad_9th_generation"],
        "ipad_10th_gen":    ["ipad_10th_generation"],
        "ipad_air_2":       ["ipad_air_2_(2014)"],
        "ipad_pro_2020":    ["ipad_pro_2020_"],
        "fire_7_2019":      ["fire_7_(2019)_"],
        "fire_hd_8_2020":   ["fire_hd_8_(2020)"],
        "fire_hd_10_2019":  ["fire_hd_10_(2019)"],
        "moto_g_3rd_gen":   ["moto_g_3rd_gen", "moto_g_3rd_gen\t",
                             "motorola_g_3rd_generation"],
        "moto_g_4th_gen":   ["moto_g_4th_gen", "moto_g4"],
        "galaxy_s4":        ["galaxy_s4",        "samsung_galaxy_s4"],
        "galaxy_s5":        ["galaxy_s5",        "samsung_galaxy_s5"],
        "galaxy_s6":        ["galaxy_s6",        "samsung_galaxy_s6"],
        "galaxy_s7_edge":   ["galaxy_s7_edge",   "samsung_galaxy_s7_edge"],
        "galaxy_s8":        ["galaxy_s8",        "samsung_galaxy_s8"],
        "galaxy_s20_fe":    ["galaxy_s20_fe",    "samsung_galaxy_s20_fe"],
        "galaxy_s21_ultra": ["galaxy_s21_ultra", "samsung_galaxy_s21_ultra"],
        "galaxy_s22_ultra": ["galaxy_s22_ultra", "samsung_galaxy_s22_ultra"],
        "pixel_3a":         ["pixel_3a",   "google_pixel_3a"],
        "pixel_4a":         ["pixel_4a",   "google_pixel_4a"],
        "pixel_4_xl":       ["pixel_4_xl", "google_pixel_4_xl"],
    }

    target_folders = gcs_folder_mapping.get(event, [event])

    frames = []
    seen_ids = set()

    for folder in target_folders:
        prefix = f"bronze/reddit/{folder}/"
        blobs  = [b for b in gcs_client.list_blobs(BUCKET_NAME, prefix=prefix)
                  if b.name.endswith(".jsonl") and "filtered" not in b.name]

        for blob in blobs:
            rows = []
            for i, line in enumerate(blob.download_as_text(encoding="utf-8").splitlines(), 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    data    = json.loads(line)
                    post_id = (data.get("post_id") or data.get("id") 
                     or data.get("name") or hash(line))
                    if post_id not in seen_ids:
                        seen_ids.add(post_id)
                        rows.append(data)
                except json.JSONDecodeError as exc:
                    log.warning("[%s][reddit] JSON error in %s line %d: %s",
                                event, blob.name, i, exc)
            if rows:
                frames.append(pd.DataFrame(rows))

    if not frames:
        log.warning("[%s][reddit] No raw Bronze blobs found across folders: %s",
                    event, target_folders)
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    log.info("[%s][reddit] Loaded %d unique rows from folders: %s",
             event, len(df), target_folders)
    return df


def process_reddit(event: str) -> int:
    """Full Bronze pipeline for Reddit: load → filter → validate → upload → bq load."""
    log.info("[%s][reddit] ── Bronze processing start ──", event)

    df = load_reddit_bronze(event)
    if df.empty:
        return 0

    n_raw = len(df)
    df    = filter_language(df, "body_text", event, "reddit")
    if df.empty:
        log.error("[%s][reddit] Empty after language filter — skipping.", event)
        return 0

    if not validate_schema(df, REQUIRED_COLS_REDDIT, event, "reddit"):
        return 0

    # Re-upload single consolidated filtered file (Bronze immutability: raw shards untouched)
    uri = upload_df_to_gcs(df, f"bronze/reddit/{event}/filtered.jsonl", fmt="jsonl")

    n_loaded = bq_load(
        source_uri=uri,
        destination_table=BQ_BRONZE_REDDIT,
        schema=SCHEMA_BRONZE_REDDIT,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        event=event,
        source="reddit",
    )

    log.info(
        "[%s][reddit] Done. raw=%d | after_filter=%d | bq_loaded=%d",
        event, n_raw, len(df), n_loaded,
    )
    return n_loaded

    # Cast days_to_launch to integer — scraper sometimes stores as float
    if "days_to_launch" in df.columns:
        df["days_to_launch"] = df["days_to_launch"].fillna(0).astype(int)

print("✓ Reddit Bronze functions defined.")

✓ Reddit Bronze functions defined.


---
## Cell 10 - Amazon Bronze processing

Amazon data has a simpler structure than Reddit - one CSV per event (produced by `amazon_data_loader.ipynb` in Stage 2), rather than daily shards.

**Source:** UCSD Amazon Reviews 2023 dataset (McAuley et al.), pre-filtered to target ASINs and the 60-day post-launch window during Stage 2. The file at `bronze/amazon/{event}.csv` is already filtered to the relevant product and time window - this cell handles language filtering and structural validation only.

**Filtered file naming:** The filtered CSV is uploaded as `{event}_filtered.csv`, not overwriting `{event}.csv`. This preserves the original Bronze file. If the language filter parameters change (e.g. the 20-character short-text threshold is adjusted), the pipeline can be re-run from this cell without re-running Stage 2.

**Expected language removal rate:** Amazon US reviews are almost entirely English. Removal rates above 5% likely indicate an ASIN mismatch (the wrong product was collected) and should be investigated before proceeding.

In [98]:
def load_amazon_bronze(event: str) -> pd.DataFrame:
    """Download Bronze JSONL for one event from GCS."""
    blob_path = f"bronze/amazon/{event}.jsonl"
    blob      = gcs_client.bucket(BUCKET_NAME).blob(blob_path)

    if not blob.exists():
        log.warning(
            "[%s][amazon] Not found: gs://%s/%s",
            event, BUCKET_NAME, blob_path,
        )
        return pd.DataFrame()

    rows = [
        json.loads(line)
        for line in blob.download_as_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    df = pd.DataFrame(rows)
    log.info("[%s][amazon] Loaded %d rows.", event, len(df))
    return df


def process_amazon(event: str) -> int:
    """Full Bronze pipeline for Amazon: load → filter → validate → upload → bq load."""
    log.info("[%s][amazon] ── Bronze processing start ──", event)

    df = load_amazon_bronze(event)
    if df.empty:
        return 0

    n_raw = len(df)
    df    = filter_language(df, "review_text", event, "amazon")
    if df.empty:
        log.error("[%s][amazon] Empty after language filter — skipping.", event)
        return 0

    if not validate_schema(df, REQUIRED_COLS_AMAZON, event, "amazon"):
        return 0

    # Separate path for filtered file — original Bronze CSV is never overwritten
    # New
    uri = upload_df_to_gcs(df, f"bronze/amazon/{event}_filtered.jsonl", fmt="jsonl")

    n_loaded = bq_load(
        source_uri=uri,
        destination_table=BQ_BRONZE_AMAZON,
        schema=SCHEMA_BRONZE_AMAZON,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        event=event,
        source="amazon",
    )

    log.info(
        "[%s][amazon] Done. raw=%d | after_filter=%d | bq_loaded=%d",
        event, n_raw, len(df), n_loaded,
    )
    return n_loaded

print("✓ Amazon Bronze functions defined.")

✓ Amazon Bronze functions defined.


# Temporary overiding for extra data
Temporarily override TARGET_EVENT to run only the failing ones


In [82]:
# Set this in a new cell above Cell 11, then run 11 → 12 → 14

events_df = events_df[events_df["product_event"].isin([
    "galaxy_tab_s2",
    "moto_g_fast",
    "moto_g_3rd_gen",
    "moto_g_4th_gen"
])]

print(f"Scoped to {len(events_df)} events: {events_df['product_event'].tolist()}")

Scoped to 4 events: ['moto_g_3rd_gen', 'moto_g_4th_gen', 'moto_g_fast', 'galaxy_tab_s2']


---
## Cell 11 - Run Bronze loading (all events)

This cell iterates over all events in `events_df` and runs the full Bronze processing pipeline for each. Events are processed sequentially. A failure in one event logs an error and continues to the next - the pipeline never aborts on a single event failure.

**Re-run safety:** Running this cell multiple times will append duplicate rows to the BigQuery Bronze tables. This is intentional — `WRITE_APPEND` plus Silver SQL deduplication means the Silver layer will always be clean regardless of how many times Bronze is loaded. If you need a clean Bronze table, truncate it manually in BigQuery before re-running.

In [132]:
events_df = events_df[events_df["product_event"].isin([
    "moto_g_4th_gen",
    "moto_g_3rd_gen",
    "moto_g_fast",
    "galaxy_tab_s2",
    "galaxy_s21_ultra",
    "galaxy_s22_ultra",
    "ipad_air_2",
])]
print(f"Scoped to {len(events_df)} events")

Scoped to 7 events


In [133]:
def process_amazon(event: str) -> int:
    """Full Bronze pipeline for Amazon — JSONL version."""
    log.info("[%s][amazon] ── Bronze processing start ──", event)

    df = load_amazon_bronze(event)
    if df.empty:
        return 0

    n_raw = len(df)
    df    = filter_language(df, "review_text", event, "amazon")
    if df.empty:
        log.error("[%s][amazon] Empty after language filter — skipping.", event)
        return 0

    if not validate_schema(df, REQUIRED_COLS_AMAZON, event, "amazon"):
        return 0

    uri = upload_df_to_gcs(df, f"bronze/amazon/{event}_filtered.jsonl", fmt="jsonl")

    n_loaded = bq_load(
        source_uri=uri,
        destination_table=BQ_BRONZE_AMAZON,
        schema=SCHEMA_BRONZE_AMAZON,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        event=event,
        source="amazon",
    )

    log.info(
        "[%s][amazon] Done. raw=%d | after_filter=%d | bq_loaded=%d",
        event, n_raw, len(df), n_loaded,
    )
    return n_loaded

print("✓ process_amazon patched to JSONL")

✓ process_amazon patched to JSONL


In [138]:
run_start = datetime.now()
log.info("══ Bronze loading start — %d event(s) ══", len(events_df))

for _, row in events_df.iterrows():
    event = row["product_event"]
    log.info("══ Event: %s ══", event)
    try:
        process_reddit(event)

        # ── Amazon inline (JSONL) ──────────────────────────────────────
        df = load_amazon_bronze(event)
        if not df.empty:
            df = filter_language(df, "review_text", event, "amazon")
        if not df.empty and validate_schema(df, REQUIRED_COLS_AMAZON, event, "amazon"):
            uri = upload_df_to_gcs(
                df, f"bronze/amazon/{event}_filtered.jsonl", fmt="jsonl"
            )
            bq_load(
                source_uri=uri,
                destination_table=BQ_BRONZE_AMAZON,
                schema=SCHEMA_BRONZE_AMAZON,
                source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
                event=event,
                source="amazon",
            )

    except Exception as exc:
        log.error("Unhandled error for event '%s': %s", event, exc)

elapsed = (datetime.now() - run_start).total_seconds()
log.info("══ Bronze loading complete — elapsed=%.1fs ══", elapsed)

2026-05-25 16:07:15  INFO      ══ Bronze loading start — 14 event(s) ══
2026-05-25 16:07:15  INFO      ══ Event: ipad_air_2 ══
2026-05-25 16:07:15  INFO      [ipad_air_2][reddit] ── Bronze processing start ──
2026-05-25 16:07:21  INFO      [ipad_air_2][reddit] Loaded 2079 unique rows from folders: ['ipad_air_2_(2014)']
2026-05-25 16:07:31  INFO      [ipad_air_2][reddit] Language filter: removed 25 of 2079 rows (1.2%)
2026-05-25 16:07:31  INFO      [ipad_air_2][reddit] Schema validation passed (2054 rows).
2026-05-25 16:07:32  INFO      Uploaded 2054 rows → gs://thesis-bucket-vua/bronze/reddit/ipad_air_2/filtered.jsonl
2026-05-25 16:07:32  INFO      [ipad_air_2][reddit] bq load: gs://thesis-bucket-vua/bronze/reddit/ipad_air_2/filtered.jsonl → vuthesis-llm-buzz.thesis_dataset.bronze_reddit
2026-05-25 16:07:34  INFO      [ipad_air_2][reddit] Load complete: appended=2054 | table_total=20249
2026-05-25 16:07:34  INFO      [ipad_air_2][reddit] Done. raw=2079 | after_filter=2054 | bq_loaded=2

---
## Cell 12 - Silver SQL transforms

Silver SQL transforms are defined in external `.sql` files and read from disk at runtime. This separation means the SQL can be edited and re-run without touching this notebook.

**What `silver_reddit.sql` does:**
1. **Deduplication** — `ROW_NUMBER() OVER (PARTITION BY post_id ORDER BY created_utc DESC)` keeps the most recent version of each post. The Reddit scraper may collect the same post across overlapping date windows for different events.
2. **Date window enforcement** — keeps only posts where `days_to_launch BETWEEN -90 AND -1` (strictly pre-launch)
3. **Short text filter** — removes rows where `ARRAY_LENGTH(SPLIT(body_text, ' ')) < 10` (fewer than 10 words)
4. **`days_to_launch` recomputation** — cross-checks the value from the scraper against `launch_date` from the events reference table

**What `silver_reviews.sql` does:**
1. **Deduplication** - `ROW_NUMBER() OVER (PARTITION BY review_id ORDER BY review_date DESC)`
2. **Date window enforcement** - keeps only reviews where `days_since_launch BETWEEN 1 AND 60`
3. **Short text filter** - removes reviews with fewer than 10 words
4. **`days_since_launch` recomputation** - validates against `launch_date`

Both transforms use `CREATE OR REPLACE TABLE` - they are fully idempotent. Re-running Silver SQL at any point produces a clean result from the current Bronze data.

In [154]:
def run_sql_file(sql_path: Path, label: str) -> None:
    """
    Read a .sql file and execute it as a BigQuery job.
    Blocks until the job completes. Raises on failure — SQL errors should
    be visible immediately, not silently swallowed.
    """
    if not sql_path.exists():
        log.error("[SQL] File not found: %s", sql_path)
        return

    sql = sql_path.read_text(encoding="utf-8").strip()
    if not sql:
        log.warning("[SQL] File is empty: %s", sql_path)
        return

    log.info("[SQL] Running %s ...", label)
    try:
        job = bq_client.query(sql)
        job.result()
        log.info("[SQL] %s complete (bytes_processed=%s).", label, job.total_bytes_processed)
    except Exception as exc:
        log.error("[SQL] %s FAILED: %s", label, exc)
        raise


log.info("══ Silver SQL transforms ══")
run_sql_file(SQL_DIR / "silver" / "silver_reddit.sql",  "silver_reddit")
run_sql_file(SQL_DIR / "silver" / "silver_reviews.sql", "silver_reviews")

2026-05-25 16:31:04  INFO      ══ Silver SQL transforms ══
2026-05-25 16:31:04  INFO      [SQL] Running silver_reddit ...
2026-05-25 16:31:07  INFO      [SQL] silver_reddit complete (bytes_processed=18352456).
2026-05-25 16:31:07  INFO      [SQL] Running silver_reviews ...
2026-05-25 16:31:09  INFO      [SQL] silver_reviews complete (bytes_processed=22878039).


---
## Cell 13 — Gold SQL aggregation

Gold tables are the final analytics-ready corpora fed directly into Stage 5 (LLM extraction). They contain one row per post/review — the same granularity as Silver — but with unnecessary columns dropped and text cleaned for LLM input.

**What `gold_pre_launch.sql` does:**
- Selects from `silver_reddit`
- Concatenates `title` and `body_text` into a single `text` field (LLMs perform better with full context)
- Joins `events` reference table to add `launch_date` and `product_type` columns
- Drops scraper metadata columns not needed for extraction (`subreddit`, `upvotes`, raw timestamps)

**What `gold_post_launch.sql` does:**
- Selects from `silver_amazon`
- Produces a `text` field from `review_text`
- Adds `rating` as a numeric ground-truth signal (used as an alternative DV in robustness checks)
- Joins `events` reference table for `product_type`

Both Gold SQL files use `CREATE OR REPLACE TABLE` — fully idempotent.

In [155]:
log.info("══ Gold SQL aggregation ══")
run_sql_file(SQL_DIR / "gold" / "gold_pre_launch.sql",  "gold_pre_launch")
run_sql_file(SQL_DIR / "gold" / "gold_post_launch.sql", "gold_post_launch")

2026-05-25 16:31:10  INFO      ══ Gold SQL aggregation ══
2026-05-25 16:31:10  INFO      [SQL] Running gold_pre_launch ...
2026-05-25 16:31:12  INFO      [SQL] gold_pre_launch complete (bytes_processed=10925069).
2026-05-25 16:31:12  INFO      [SQL] Running gold_post_launch ...
2026-05-25 16:31:15  INFO      [SQL] gold_post_launch complete (bytes_processed=2681318).


---
## Cell 14 — Threshold checks

After Silver transforms have run, this cell queries the Silver tables to verify that each product event meets the minimum corpus size requirements defined in the research design:

- **Pre-launch (Reddit):** ≥ 500 posts per event
- **Post-launch (Amazon):** ≥ 200 reviews per event

These thresholds were set to ensure sufficient statistical power for the OLS regression in Stage 7. With ~8–10 events and ~32–40 observations in the final long-format dataframe, events below the minimum threshold risk being driven by outlier posts or a handful of reviews rather than stable aggregate signals.

**What to do if an event fails a threshold:**
- Check the collection scripts (Stage 1/2) for that event — the scraper may have stopped early or hit a rate limit
- Consider whether the event should be replaced with an alternative product launch
- Do not proceed to Stage 5 for events below threshold — extraction from thin corpora produces unreliable signal estimates

Events present in `events.csv` but missing from Silver tables entirely (no rows loaded) are flagged separately — this typically means the Bronze file was never collected.

In [156]:
def check_thresholds() -> None:
    """Query Silver tables and log pass/fail for each event against minimum thresholds."""
    log.info("══ Threshold checks ══")

    pre_query  = f"SELECT product_event, COUNT(*) AS n FROM `{BQ_SILVER_REDDIT}` GROUP BY product_event ORDER BY product_event"
    post_query = f"SELECT product_event, COUNT(*) AS n FROM `{BQ_SILVER_AMAZON}` GROUP BY product_event ORDER BY product_event"

    try:
        pre_df  = bq_client.query(pre_query).to_dataframe()
        post_df = bq_client.query(post_query).to_dataframe()
    except Exception as exc:
        log.error("Threshold queries failed (Silver tables may not exist yet): %s", exc)
        return

    all_events = set(events_df["product_event"])

    for _, row in pre_df.iterrows():
        status = "OK" if row["n"] >= MIN_PRE_LAUNCH else "BELOW THRESHOLD"
        log.info("  [pre ]  %-22s  n=%5d  (min=%d)  %s",
                 row["product_event"], row["n"], MIN_PRE_LAUNCH, status)

    for _, row in post_df.iterrows():
        status = "OK" if row["n"] >= MIN_POST_LAUNCH else "BELOW THRESHOLD"
        log.info("  [post]  %-22s  n=%5d  (min=%d)  %s",
                 row["product_event"], row["n"], MIN_POST_LAUNCH, status)

    pre_set  = set(pre_df["product_event"])  if not pre_df.empty  else set()
    post_set = set(post_df["product_event"]) if not post_df.empty else set()

    for ev in sorted(all_events - pre_set):
        log.warning("  [pre ]  %-22s  MISSING from silver_reddit entirely!", ev)
    for ev in sorted(all_events - post_set):
        log.warning("  [post]  %-22s  MISSING from silver_amazon entirely!", ev)

    # Return summary DataFrames for display in notebook
    return pre_df, post_df


pre_counts, post_counts = check_thresholds()

print("\n── Pre-launch (Reddit) ──")
display(pre_counts.assign(threshold=MIN_PRE_LAUNCH, meets_threshold=pre_counts["n"] >= MIN_PRE_LAUNCH))

print("── Post-launch (Amazon) ──")
display(post_counts.assign(threshold=MIN_POST_LAUNCH, meets_threshold=post_counts["n"] >= MIN_POST_LAUNCH))

2026-05-25 16:31:20  INFO      ══ Threshold checks ══


c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


2026-05-25 16:31:21  INFO        [pre ]  galaxy_s20_fe           n= 2754  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  galaxy_s21_ultra        n=  769  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  galaxy_s22_ultra        n= 3713  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  galaxy_s6               n= 5269  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  galaxy_tab_s2           n=  599  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  ipad_air_2              n= 1758  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  ipad_air_m1             n= 1361  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  ipad_pro_2018           n= 2906  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  lg_g3                   n= 2802  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  moto_g_3rd_gen          n=  674  (min=500)  OK
2026-05-25 16:31:21  INFO        [pre ]  moto_g_4th_gen          n=  190  (min=500)  BELOW THRESHOLD
2026-05-25 16:31:21

c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_event,n,threshold,meets_threshold
0,galaxy_s20_fe,2754,500,True
1,galaxy_s21_ultra,769,500,True
2,galaxy_s22_ultra,3713,500,True
3,galaxy_s6,5269,500,True
4,galaxy_tab_s2,599,500,True
5,ipad_air_2,1758,500,True
6,ipad_air_m1,1361,500,True
7,ipad_pro_2018,2906,500,True
8,lg_g3,2802,500,True
9,moto_g_3rd_gen,674,500,True


── Post-launch (Amazon) ──


,product_event,n,threshold,meets_threshold
0,galaxy_s20_fe,224,200,True
1,galaxy_s21_ultra,190,200,False
2,galaxy_s22_ultra,250,200,True
3,galaxy_s6,204,200,True
4,galaxy_tab_s2,1061,200,True
5,ipad_air_2,198,200,False
6,ipad_air_m1,213,200,True
7,ipad_pro_2018,317,200,True
8,lg_g3,214,200,True
9,moto_g_3rd_gen,304,200,True


---
## Cell 15 — Upload run log to GCS

The final step uploads the local log file to `gs://thesis-bucket/logs/` for permanent storage. This log constitutes the audit trail for this ETL run — it records every transformation applied, every row count before and after each step, every BigQuery job outcome, and every threshold check result.

For thesis reproducibility documentation, the logs directory in GCS provides a complete record of how the Silver and Gold tables were produced, including timestamps and intermediate row counts. This is particularly important for the methodology section — the log confirms the language filter removal rates and the pre/post-launch corpus sizes reported in the thesis.

In [157]:
if LOG_FILE_PATH.exists():
    blob_path = f"logs/{LOG_FILE_PATH.name}"
    gcs_client.bucket(BUCKET_NAME).blob(blob_path).upload_from_filename(str(LOG_FILE_PATH))
    log.info("Log uploaded → gs://%s/%s", BUCKET_NAME, blob_path)
else:
    print("Log file not found — skipping upload.")

2026-05-25 16:31:56  INFO      Log uploaded → gs://thesis-bucket-vua/logs/orchestrate_20260525_134743.log


---
## Summary: what this notebook produces

| BigQuery table | Contents | Next stage |
|---|---|---|
| `bronze_reddit` | Raw Reddit posts, language-filtered, all events combined | Input for Silver SQL |
| `bronze_amazon` | Raw Amazon reviews, language-filtered, all events combined | Input for Silver SQL |
| `silver_reddit` | Deduplicated, date-windowed, short-text-filtered Reddit posts | Input for Gold SQL + EDA |
| `silver_amazon` | Deduplicated, date-windowed, short-text-filtered Amazon reviews | Input for Gold SQL + EDA |
| `gold_pre_launch` | Clean pre-launch corpus with concatenated text field | Stage 5 — LLM extraction (IV1, IV2, IV3) |
| `gold_post_launch` | Clean post-launch corpus with review text and star rating | Stage 5 — ABSA extraction (DV) |

**Proceed to `pre_launch_buzz_eda.ipynb` and `post_launch_reviews_eda.ipynb` for Stage 4b EDA.**